# Notebook 03 — NLP (TF-IDF + Ridge Model)

**Requires**: Notebook 02 outputs in `../processed_features/`  
**Goal**: Train a pure text-based model using TF-IDF and Ridge Regression to serve as a strong baseline and a meta-feature for down-stream tree models.

**Strategy**:
1. Clean and combine Title, Bullet Points, and Description.
2. Fit a single `TfidfVectorizer` (up to 100k features, word ngrams (1, 2)).
3. Train two Ridge models:
   - **Model A**: Predicts `log1p(PRODUCT_LENGTH)` using Ridge.
   - **Model B**: Predicts raw `PRODUCT_LENGTH` using sample weights optimized for MAPE ($1 / y$).
4. Blend Model A and Model B predictions.
5. Save predictions to `../processed_features/nlp_train/val/test.npy`.

## 0. DATA MODE

In [1]:
# ============================================================
# DATA MODE: Changed to experiment mode
# ============================================================
DATA_MODE = "experiment"
DATA_PATHS = {
    "debug":      "../dataset/sampled/debug",
    "experiment": "../dataset/sampled/experiment",
    "full":       "../dataset"
}
DATA_DIR = DATA_PATHS[DATA_MODE]
print("Using dataset:", DATA_DIR)


Using dataset: ../dataset/sampled/experiment


## 1. Imports & Helpers

In [2]:
import os, re, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

def mape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask = y_true > 0
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / y_true[mask]))

def clean_text(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower().strip()
    text = re.sub(r'[\u00d7\u2715]', ' x ', text)
    text = re.sub(r'[^\x00-\x7f]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

print("✅ Imports & helpers done")


✅ Imports & helpers done


## 2. Load Data

In [3]:
df_train_all = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
df_test      = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

train_idx = np.load("../processed_features/train_indices.npy")
val_idx   = np.load("../processed_features/val_indices.npy")

df_train = df_train_all.iloc[train_idx].reset_index(drop=True)
df_val   = df_train_all.iloc[val_idx].reset_index(drop=True)

y_train = df_train['PRODUCT_LENGTH'].values.astype(float)
y_val   = df_val['PRODUCT_LENGTH'].values.astype(float)

print(f"Train size: {len(df_train):,} | Val size: {len(df_val):,}")


Train size: 160,000 | Val size: 40,000


## 3. TF-IDF Representation

In [4]:
def get_combined_text(df):
    t = df['TITLE'].fillna('').apply(clean_text)
    b = df['BULLET_POINTS'].fillna('').apply(clean_text)
    d = df['DESCRIPTION'].fillna('').apply(clean_text)
    return (t + ' ' + b + ' ' + d).str.strip()

print("Combining text fields...")
text_tr = get_combined_text(df_train)
text_va = get_combined_text(df_val)
text_te = get_combined_text(df_test)

print("Fitting TF-IDF Vectorizer (ngram_range=(1, 2), max_features=100,000)...")
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100_000, sublinear_tf=True)
X_tr = tfidf.fit_transform(text_tr)
X_va = tfidf.transform(text_va)
X_te = tfidf.transform(text_te)

print(f"TF-IDF Matrix shape: {X_tr.shape}")


Combining text fields...
Fitting TF-IDF Vectorizer (ngram_range=(1, 2), max_features=100,000)...
TF-IDF Matrix shape: (160000, 100000)


## 4. Train NLP Models

In [5]:
# --- Model A: Ridge on log1p Target ---
print("Training Ridge A (log1p target, alpha=1.0)...")
ridge_A = Ridge(alpha=1.0, random_state=42)
ridge_A.fit(X_tr, np.log1p(y_train))

pA_tr = np.clip(np.expm1(ridge_A.predict(X_tr)), 0.5, None)
pA_va = np.clip(np.expm1(ridge_A.predict(X_va)), 0.5, None)
pA_te = np.clip(np.expm1(ridge_A.predict(X_te)), 0.5, None)

print(f"Model A Train MAPE: {mape(y_train, pA_tr)*100:.2f}%")
print(f"Model A Val MAPE:   {mape(y_val, pA_va)*100:.2f}%")

# --- Model B: Weighted Ridge on Raw Target ---
print("\nTraining Ridge B (weighted raw target, alpha=10.0)...")
w_tr = 1.0 / np.maximum(y_train, 10.0)
w_tr /= w_tr.mean()

ridge_B = Ridge(alpha=10.0, random_state=42)
ridge_B.fit(X_tr, y_train, sample_weight=w_tr)

pB_tr = np.clip(ridge_B.predict(X_tr), 0.5, None)
pB_va = np.clip(ridge_B.predict(X_va), 0.5, None)
pB_te = np.clip(ridge_B.predict(X_te), 0.5, None)

print(f"Model B Train MAPE: {mape(y_train, pB_tr)*100:.2f}%")
print(f"Model B Val MAPE:   {mape(y_val, pB_va)*100:.2f}%")


Training Ridge A (log1p target, alpha=1.0)...
Model A Train MAPE: 69.12%
Model A Val MAPE:   155.13%

Training Ridge B (weighted raw target, alpha=10.0)...
Model B Train MAPE: 66.66%
Model B Val MAPE:   117.82%


## 5. Optimize Blend & Save NLP Meta-Features

In [6]:
best_w, best_m = 0.5, float('inf')
for w in np.arange(0.0, 1.05, 0.05):
    blend_va = w * pA_va + (1.0 - w) * pB_va
    m = mape(y_val, blend_va)
    if m < best_m:
        best_m = m
        best_w = w

print(f"Best blend weight (Model A): {best_w:.2f} | Val MAPE: {best_m*100:.2f}%")

nlp_tr = best_w * pA_tr + (1.0 - best_w) * pB_tr
nlp_va = best_w * pA_va + (1.0 - best_w) * pB_va
nlp_te = best_w * pA_te + (1.0 - best_w) * pB_te

np.save("../processed_features/nlp_train.npy", nlp_tr)
np.save("../processed_features/nlp_val.npy",   nlp_va)
np.save("../processed_features/nlp_test.npy",  nlp_te)
print("✅ NLP Predictions saved to nlp_*.npy")


Best blend weight (Model A): 0.00 | Val MAPE: 117.82%
✅ NLP Predictions saved to nlp_*.npy
